<a href="https://colab.research.google.com/github/svyatoslavna/ml_hw/blob/main/w2v_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [5]:
!pip install gensim -q

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [6]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

*Ответила в коде!*

In [7]:
api.info(name_only=True) # датасеты

{'corpora': ['semeval-2016-2017-task3-subtaskBC',
  'semeval-2016-2017-task3-subtaskA-unannotated',
  'patent-2017',
  'quora-duplicate-questions',
  'wiki-english-20171001',
  'text8',
  'fake-news',
  '20-newsgroups',
  '__testing_matrix-synopsis',
  '__testing_multipart-matrix-synopsis'],
 'models': ['fasttext-wiki-news-subwords-300',
  'conceptnet-numberbatch-17-06-300',
  'word2vec-ruscorpora-300',
  'word2vec-google-news-300',
  'glove-wiki-gigaword-50',
  'glove-wiki-gigaword-100',
  'glove-wiki-gigaword-200',
  'glove-wiki-gigaword-300',
  'glove-twitter-25',
  'glove-twitter-50',
  'glove-twitter-100',
  'glove-twitter-200',
  '__testing_word2vec-matrix-synopsis']}

In [8]:
datasets = ['text8', 'word2vec-ruscorpora-300', 'fake-news']

for dataset in datasets:
    info = api.info(dataset)
    print(f'Датасет: {dataset}')
    print(f'Количество строк: {info['num_records']}')
    print(f'Описание: {info['description']}') # про источник и примерный объём тоже
    print()

print('"text8" - для обучения word embeddings, тестирования моделей на '
      'стандартом неспецифическом корпусе, подходит для начала обучения NLP;')
print('"word2vec-ruscorpora-300" - это уже готовые векторы, можно применить для'
      ' решения NLP-задач на русском языке, семантического анализа (например, '
      'тематический анализ);')
print('"fake-news" - можно изучить феномен fake-news, определить особый язык, '
      ' стилистику fake-news, научить модель отличать потенциально '
      'недостоверную информацию от достоверной - потребуется датасет '
      'с правдивыми новостями.')

Датасет: text8
Количество строк: 1701
Описание: First 100,000,000 bytes of plain text from Wikipedia. Used for testing purposes; see wiki-english-* for proper full Wikipedia datasets.

Датасет: word2vec-ruscorpora-300
Количество строк: 184973
Описание: Word2vec Continuous Skipgram vectors trained on full Russian National Corpus (about 250M words). The model contains 185K words.

Датасет: fake-news
Количество строк: 12999
Описание: News dataset, contains text and metadata from 244 websites and represents 12,999 posts in total from a specific window of 30 days. The data was pulled using the webhose.io API, and because it's coming from their crawler, not all websites identified by their BS Detector are present in this dataset. Data sources that were missing a label were simply assigned a label of 'bs'. There are (ostensibly) no genuine, reliable, or trustworthy news sources represented in this dataset (so far), so don't trust anything you read.

"text8" - для обучения word embeddings, тес

**Базовые операции с векторами**

In [10]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [11]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


*Ваш ответ здесь*

**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [17]:
my_w2v = api.load('word2vec-ruscorpora-300')

2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [113]:
def find_similar_ru(word):
    """
    Находит и выводит 10 слов, наиболее похожих слов.
    Используется предобученная модель my_w2v (word2vec).
    Результаты выводятся в консоль в виде нумерованного списка.

    Args:
        word (str): Слово в формате 'слово_ЧАСТЬ_РЕЧИ' (например, 'трезво_ADV',
                   'коммунизм_NOUN', 'реванш_NOUN', 'зычный_ADJ')

    Returns:
        None: Ничего не возвращает, только выводит результаты в консоль

    Prints:
        10 наиболее близких по вектору слов для указанного слова с их оценками
        схожести
    """
    res_words = my_w2v.most_similar(word, topn=10)

    print(f'10 наиболее близких по вектору слов для слова {word}:')
    for i, w in enumerate(res_words):
        print(f'{i+1}. {w[0]}: {w[1]:.4f}')

    print()

In [114]:
find_similar_ru('трезво_ADV')
find_similar_ru('коммунизм_NOUN')
find_similar_ru('реванш_NOUN')
find_similar_ru('зычный_ADJ')

10 наиболее близких по вектору слов для слова трезво_ADV:
1. здраво_ADV: 0.5711
2. критически_ADV: 0.5319
3. разумно_ADV: 0.5224
4. объективно_ADV: 0.4970
5. непредвзято_ADV: 0.4935
6. оценивать_VERB: 0.4824
7. беспристрастно_ADV: 0.4808
8. вдумчиво_ADV: 0.4728
9. хладнокровно_ADV: 0.4701
10. серьезно_ADV: 0.4699

10 наиболее близких по вектору слов для слова коммунизм_NOUN:
1. социализм_NOUN: 0.8398
2. коммунистический_ADJ: 0.7232
3. капитализм_NOUN: 0.6762
4. диктатура::пролетариат_NOUN: 0.6338
5. социалистический_ADJ: 0.6004
6. бесклассовый_ADJ: 0.5942
7. фашизм_NOUN: 0.5914
8. марксизм_NOUN: 0.5882
9. ленинизм_NOUN: 0.5851
10. капиталистический_ADJ: 0.5843

10 наиболее близких по вектору слов для слова реванш_NOUN:
1. касымжан_NOUN: 0.4773
2. размен::ферзь_NOUN: 0.4701
3. широв_NOUN: 0.4578
4. ллеида_NOUN: 0.4543
5. дреев_NOUN: 0.4532
6. лотье_NOUN: 0.4463
7. виши::ананд_NOUN: 0.4434
8. победа_NOUN: 0.4383
9. ромарио_NOUN: 0.4369
10. чемпионский::титул_NOUN: 0.4336

10 наиболее бли

3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [45]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [46]:
sg_model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    sg=1
)

In [47]:
print(f"Слова в словаре: {list(sg_model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [115]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = sg_model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  вино: 0.2398
  ингредиенты: 0.2172
  хлеб: 0.1938
  брокколи: 0.1846
  кипятить: 0.1711


In [142]:
def find_similar(model, word):
    """
    Выводит 5 слов, наиболее похожих на заданное.
    Используется обученная модель (заданная).
    Результаты выводятся в консоль с указанием степени схожести (косинусной близости).

    Args:
        model (Word2Vec or FastText): Обученная модель для работы с векторными представлениями.
        word (str): Слово для поиска похожих слов (в формате, соответствующем словарю модели,
                   например, 'духовка', 'трезво_ADV' и т.д.)

    Returns:
        None: Функция ничего не возвращает, только выводит результаты в консоль.

    Prints:
        Заголовок с искомым словом и список из 5 наиболее похожих слов с их оценками схожести.
        Если слово не найдено в словаре модели, выводится сообщение об ошибке.

    Raises:
        KeyError: Обрабатывается внутри функции, внешних исключений не вызывает
    """

    try:
        print(f"\nСлова, похожие на '{word}':")
        similar = model.wv.most_similar(word, topn=5)
        for lex, score in similar:
            print(f"  {lex}: {score:.4f}")
    except KeyError:
        print(f"Слово '{lex}' не найдено в словаре")

In [143]:
find_similar(sg_model, 'духовка')
find_similar(sg_model, 'овощи')


Слова, похожие на 'духовка':
  ингредиенты: 0.3199
  десерт: 0.3064
  холодильник: 0.2705
  питание: 0.2243
  пирог: 0.2142

Слова, похожие на 'овощи':
  мариновать: 0.2716
  хлеб: 0.2691
  гриль: 0.2546
  фольга: 0.2409
  сахар: 0.2108


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [79]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [118]:
find_similar(ft_model, 'варить')
find_similar(ft_model, 'духовка')
find_similar(ft_model, 'овощи')


Слова, похожие на 'варить':
  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
  тушить: 0.3405
  специи: 0.2622

Слова, похожие на 'духовка':
  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
  курица: 0.3041
  тост: 0.2944

Слова, похожие на 'овощи':
  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297
  соус: 0.2172
  торт: 0.2094


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [84]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = sg_model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('learning')
compare_models('neural')


Сравнение для слова: 'learning'
  Word2Vec: слово не найдено
  FastText: ['духовка', 'пирог']

Сравнение для слова: 'neural'
  Word2Vec: слово не найдено
  FastText: ['мука', 'травы']


In [89]:
compare_models('жарррить')
compare_models('гатовит')
compare_models('топт')


Сравнение для слова: 'жарррить'
  Word2Vec: слово не найдено
  FastText: ['жарить', 'варить']

Сравнение для слова: 'гатовит'
  Word2Vec: слово не найдено
  FastText: ['яйца', 'сливки']

Сравнение для слова: 'топт'
  Word2Vec: слово не найдено
  FastText: ['кофе', 'десерт']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [94]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [95]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [96]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [99]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [104]:
compare_documents(2, 4)

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [110]:
doc_model.dv.most_similar("doc_1", topn=1) # это doc_0

[('doc_0', 0.2735169529914856)]

In [111]:
compare_documents(1, 0) # поподробнее про пару doc_0 и doc_1

Схожесть doc_1 и doc_0: 0.2735
  doc_1: deep learning uses neural networks
  doc_0: machine learning is interesting


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [119]:
sizes = [10, 50, 100]
models = []

for size in sizes:
    model = FastText(
    sentences=cooking_sentences,
    vector_size=size,
    window=3,
    min_count=1,
    workers=2
)

    models.append(model)

In [139]:
words = ['холодильник', 'дрожжи', 'барбекю']

for i, model in enumerate(models):
    print(f'\n=== ТЕСТ модели с размерностью {sizes[i]} ===')
    for word in words:
        find_similar(model, word)
    print('=====================================\n')


=== ТЕСТ модели с размерностью 10 ===

Слова, похожие на 'холодильник':
  говядина: 0.7501
  яичница: 0.7252
  торт: 0.6424
  соус: 0.5909
  яйца: 0.5171

Слова, похожие на 'дрожжи':
  уголь: 0.5751
  соль: 0.5668
  кипятить: 0.5258
  сковорода: 0.5150
  сливки: 0.5005

Слова, похожие на 'барбекю':
  пирог: 0.6012
  тушить: 0.4551
  специи: 0.4511
  горшок: 0.4480
  хлеб: 0.4425


=== ТЕСТ модели с размерностью 50 ===

Слова, похожие на 'холодильник':
  яблоки: 0.3735
  дрожжи: 0.2327
  говядина: 0.2294
  салат: 0.1949
  тесто: 0.1880

Слова, похожие на 'дрожжи':
  яйца: 0.4264
  готовить: 0.3810
  тесто: 0.2737
  чай: 0.2643
  вино: 0.2585

Слова, похожие на 'барбекю':
  рыба: 0.2724
  мясо: 0.2247
  соль: 0.2209
  парить: 0.2076
  тушить: 0.1858


=== ТЕСТ модели с размерностью 100 ===

Слова, похожие на 'холодильник':
  варить: 0.2333
  огурцы: 0.2192
  резать: 0.1696
  яблоки: 0.1532
  помидоры: 0.1407

Слова, похожие на 'дрожжи':
  уголь: 0.1467
  соль: 0.1397
  помидоры: 0.1350


Я попыталась оценить полученные результаты, опираясь на своё собственное ощущение семантической близости данных слов. Получаются следующие результаты (за каждое, семантически-близкое из подобранных моделью слов с заданным, ставлю 1 балл, т.е. одна модель может получить максимум 15 баллов):

  - Модель с vector_size=10:  7 баллов
  - Модель с vector_size=50: **10 баллов**
  - Модель с vector_size=100: 6 баллов

Модель среднего размера справилась лучше всех. Первая, скорее всего, из-за маленькой размерности недообучилась, третья - переобучилась. У данного эксперимента есть ограничение - оно состоит в очень небольшом размере датасета.